# 14 · SAMBA-UNet: Python API and Hydra CLI

SAMBA-UNet (`sam3_fine_tune/sambaunet/`) is the semantic-segmentation baseline we compare against the SAM3 fine-tune.
This notebook is **documentation**, not an nbdev export source (`settings.ini` was never migrated, see `CLAUDE.md`).

It runs top to bottom on a tiny **synthetic** COCO dataset generated below (no private data, no downloads: the SAM
encoder is the random `sam_tiny` variant), on CPU, in about a minute. Two parts:

1. **Python API**: `CocoSemanticDataset` -> `build_model` -> training steps with `bce_dice_loss` -> predict -> `CategoryScores` -> `load_model`.
2. **CLI**: the same config via `python -m sam3_fine_tune.sambaunet.train` / `.evaluate`, and how to read `multiclass_foreground_iou_summary.json`.

Run it from the repo root or from `nbs/`. The numbers are meaningless as quality (random encoder, 3 epochs); the point is that both routes agree.

In [ ]:
import json, subprocess, sys, tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader

REPO = Path.cwd().parent if Path.cwd().name == "nbs" else Path.cwd()
WORK = Path(tempfile.mkdtemp(prefix="sambaunet_nb_"))   # all outputs live here; paths are never printed
print("repo root found:", (REPO / "sam3_fine_tune" / "sambaunet").is_dir())

## 0 · A tiny synthetic COCO dataset
Standard layout `<root>/{train,test}/{images,annotation.json}`. Category `"bar"` is a horizontal rectangle,
`"disc"` an ellipse; `category_name` doubles as the SAM3 text prompt, so the same dataset also feeds the SAM3 pipeline.

In [ ]:
CATS = [{"id": 1, "name": "bar"}, {"id": 2, "name": "disc"}]

def make_split(root: Path, split: str, n: int, seed: int, size: int = 96):
    rng = np.random.default_rng(seed)
    (root / split / "images").mkdir(parents=True)
    images, anns = [], []
    for i in range(n):
        img = Image.fromarray(rng.integers(30, 80, (size, size, 3), dtype=np.uint8))
        d = ImageDraw.Draw(img)
        x, y = (int(v) for v in rng.integers(4, size // 2, 2))
        w, h = (int(v) for v in rng.integers(20, size // 2 - 4, 2))
        d.rectangle([x, y, x + w, y + h // 3], fill=(230, 60, 60))                       # bar
        cx, cy = (int(v) for v in rng.integers(size // 2 + 8, size - 14, 2)); r = 10
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(60, 220, 90))                   # disc
        img.save(root / split / "images" / f"{i:03d}.png")
        images.append({"id": i, "file_name": f"{i:03d}.png", "width": size, "height": size})
        anns.append({"id": 2 * i, "image_id": i, "category_id": 1, "iscrowd": 0, "area": w * h // 3, "bbox": [x, y, w, h // 3],
                     "segmentation": [[x, y, x + w, y, x + w, y + h // 3, x, y + h // 3]]})
        pts = [c for a in np.linspace(0, 2 * np.pi, 16, endpoint=False) for c in (cx + r * np.cos(a), cy + r * np.sin(a))]
        anns.append({"id": 2 * i + 1, "image_id": i, "category_id": 2, "iscrowd": 0, "area": 314, "bbox": [cx - r, cy - r, 2 * r, 2 * r],
                     "segmentation": [pts]})
    (root / split / "annotation.json").write_text(json.dumps({"images": images, "annotations": anns, "categories": CATS}))

DATA = WORK / "dataset"
make_split(DATA, "train", 8, seed=0)
make_split(DATA, "test", 4, seed=1)
print(sorted(p.name for p in DATA.iterdir()), len(list((DATA / "train" / "images").iterdir())), "train images")

## 1 · Python API
### 1.1 Config
We load the shipped `conf/config.yaml` and apply the same overrides the CLI will get in part 2 (`OmegaConf.from_dotlist`),
so both routes share one definition of "the config". `sam_tiny=true` swaps ViT-B for a random tiny encoder (no Hub download).

In [ ]:
OVERRIDES = ["model.sam_tiny=true", "model.dims=[8,16,32,64]", "model.grad_checkpoint=true",
             "data.size=64", "data.batch_size=2", "data.num_workers=0", "data.augment=false",
             "trainer.epochs=3", "trainer.device=cpu", "trainer.log_every=1"]
cfg = OmegaConf.merge(OmegaConf.load(REPO / "sam3_fine_tune/sambaunet/conf/config.yaml"),
                      OmegaConf.from_dotlist(OVERRIDES + [f"data.root={DATA}"]))
del cfg["hydra"]   # `${now:...}` run-dir interpolation only resolves inside the Hydra CLI; not needed here
print(OmegaConf.to_yaml(cfg.model), OmegaConf.to_yaml(cfg.trainer))

### 1.2 Data: `CocoSemanticDataset`
Channel *k* of the mask is category *k*; images and masks are resized to `data.size`.

In [ ]:
from sam3_fine_tune.sambaunet.data import CocoSemanticDataset

train_ds = CocoSemanticDataset(DATA / "train", cfg.data.size, cfg.data.augment, ann_file=cfg.data.ann_file)
val_ds = CocoSemanticDataset(DATA / "test", cfg.data.size, False, train_ds.category_names, cfg.data.ann_file)
item = train_ds[0]
print(train_ds.category_names, item["image"].shape, item["mask"].shape, "foreground px per channel:", item["mask"].sum((1, 2)).tolist())

### 1.3 Model and a few training steps
`build_model` reads `cfg.model`; the frozen SAM encoder contributes no trainable parameters (only LoRA/adapters/Mamba U-Net).

In [ ]:
from sam3_fine_tune.sambaunet.checkpoint import build_model, save_checkpoint
from sam3_fine_tune.sambaunet.model import bce_dice_loss
from sam3_fine_tune.sambaunet.train import resolve_device, validate

import random
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)   # same seeding as the CLI's fit()
device = resolve_device(cfg.trainer.device)
model = build_model(cfg.model, len(train_ds.category_names)).to(device)
params = [p for p in model.parameters() if p.requires_grad]
print(f"trainable {sum(p.numel() for p in params)/1e3:.0f}k of {sum(p.numel() for p in model.parameters())/1e3:.0f}k params")

kw = dict(batch_size=cfg.data.batch_size, num_workers=cfg.data.num_workers)
train_dl = DataLoader(train_ds, shuffle=True, drop_last=len(train_ds) >= cfg.data.batch_size, **kw)
val_dl = DataLoader(val_ds, **kw)
opt = torch.optim.AdamW(params, lr=cfg.optim.lr, weight_decay=cfg.optim.weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.trainer.epochs * len(train_dl), eta_min=cfg.optim.min_lr)

for epoch in range(cfg.trainer.epochs):
    model.train()
    for batch in train_dl:
        loss = bce_dice_loss(model(batch["image"].to(device)), batch["mask"].to(device))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    res = validate(model, val_dl, device, train_ds.category_names, cfg.trainer.threshold)
    print(f"epoch {epoch}  last-batch loss {loss.item():.4f}  val macro FG IoU {res['macro_foreground_iou']:.2f}")
py_best = res

### 1.4 Checkpoint round-trip: `save_checkpoint` / `load_model`
Checkpoints hold only trainable weights + the config, so `load_model` rebuilds the architecture itself (and loads with `weights_only=True`).

In [ ]:
from sam3_fine_tune.sambaunet.checkpoint import load_model

ckpt_path = WORK / "py_last.pt"
save_checkpoint(ckpt_path, model, cfg, train_ds.category_names, epoch=cfg.trainer.epochs - 1, val=py_best)
loaded, ck = load_model(ckpt_path, device)
x = val_ds[0]["image"].unsqueeze(0).to(device)
with torch.no_grad():
    same = torch.allclose(model.eval()(x), loaded(x), atol=1e-5)
print("categories in checkpoint:", ck["category_names"], "| reloaded model reproduces logits:", same)

### 1.5 Predict and score with `CategoryScores`
As in the evaluate CLI: predict at training resolution, upsample logits to the native image size, score against the original COCO masks.

In [ ]:
import torch.nn.functional as F
from sam3_fine_tune.sambaunet.metrics import CategoryScores

scores, preds = CategoryScores(val_ds.category_names), []
with torch.no_grad():
    for i in range(len(val_ds)):
        gt = val_ds.original_mask(i)
        logits = loaded(val_ds[i]["image"].unsqueeze(0).to(device))
        logits = F.interpolate(logits, gt.shape[-2:], mode="bilinear", align_corners=False)
        pred = (torch.sigmoid(logits)[0] > cfg.trainer.threshold).cpu().numpy()
        scores.update(pred, gt, val_ds.image_path(i).name); preds.append(pred)
py_summary = scores.summary()
print({n: c["overall_foreground_iou"] for n, c in py_summary["categories"].items()}, "| overall", py_summary["overall"]["foreground_iou"])

### 1.6 Prediction overlays
Left: ground truth, right: prediction (red = `bar`, green = `disc`). With a random encoder and 3 epochs expect rough blobs.

In [ ]:
from sam3_fine_tune.sambaunet.evaluate import PALETTE

def blend(path, masks):
    img = np.asarray(Image.open(path).convert("RGB")).astype(np.float32)
    for k, m in enumerate(masks.astype(bool)):
        img[m] = 0.5 * img[m] + 0.5 * PALETTE[k % len(PALETTE)]
    return img.astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(7, 4.6))
for j, ax in enumerate(axes.T):
    ax[0].imshow(blend(val_ds.image_path(j), val_ds.original_mask(j))); ax[0].set_title("ground truth", fontsize=8)
    ax[1].imshow(blend(val_ds.image_path(j), preds[j])); ax[1].set_title("prediction", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 2 · Hydra CLI
The same run as shell commands. In a terminal (repo root):

```bash
python -m sam3_fine_tune.sambaunet.train data.root=/path/to/dataset trainer.epochs=3 model.grad_checkpoint=true
python -m sam3_fine_tune.sambaunet.evaluate checkpoint=experiments/sambaunet/<run>/checkpoints/best.pt data.root=/path/to/dataset save_overlays=true
```

Any key in `conf/config.yaml` / `conf/eval.yaml` is overridable as `key=value`; Hydra writes the resolved config to `.hydra/` in the run directory.
Below we call them from the notebook with `subprocess` (equivalent to `!python -m ...`) using the **same `OVERRIDES`** as part 1;
only `hydra.run.dir` is added to keep outputs in the scratch dir. A small helper redacts the scratch path from the logs.

In [ ]:
def run_cli(module, *args):
    cmd = [sys.executable, "-m", f"sam3_fine_tune.sambaunet.{module}", *args]
    p = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (p.stdout + p.stderr).replace(str(WORK), "<WORK>").replace(str(REPO), "<REPO>")
    assert p.returncode == 0, out[-3000:]
    return out

RUN = WORK / "cli_run"
log = run_cli("train", f"data.root={DATA}", f"hydra.run.dir={RUN}", *OVERRIDES)
print("\n".join(l for l in log.splitlines() if "epoch" in l or "trainable" in l))
print(sorted(p.name for p in (RUN / "checkpoints").iterdir()))

In [ ]:
EVAL = WORK / "cli_eval"
log = run_cli("evaluate", f"checkpoint={RUN / 'checkpoints' / 'last.pt'}", f"data.root={DATA}", "device=cpu",
              "save_overlays=true", f"hydra.run.dir={EVAL}")
print("\n".join(l for l in log.splitlines() if "FG IoU" in l))
print(sorted(p.name for p in EVAL.iterdir() if not p.name.startswith(".")), len(list((EVAL / "overlays").iterdir())), "overlays")

### Reading `multiclass_foreground_iou_summary.json`
Same layout as the SAM3 multi-class evaluation (`evaluation/multi_eval_pipe.py`), plus a `meta` block:

* `summary.categories.<name>`: `overall_foreground_iou` (pixel-aggregated TP/(TP+FP+FN)), `mean_foreground_iou` (mean of per-image IoU; empty-vs-empty = 1.0), `pixel_precision`, `pixel_recall`, `pixel_f1`, `best_sample` / `worst_sample`.
* `summary.overall`: the same pixel counts pooled over all categories; `summary.macro_foreground_iou` is the mean over categories (extra key, not in the SAM3 file).
* `per_category_details.<name>.per_image_results`: one row per image, for error analysis.
* `meta`: trainable/total params, input size, threshold, median latency per image.

In [ ]:
res = json.loads((EVAL / "multiclass_foreground_iou_summary.json").read_text())
for name, c in res["summary"]["categories"].items():
    print(f"{name:6s} IoU {c['overall_foreground_iou']:6.2f}  P {c['pixel_precision']:6.2f}  R {c['pixel_recall']:6.2f}  worst {c['worst_sample']['file_name']}")
print("overall", res["summary"]["overall"]["foreground_iou"], "| macro", res["summary"]["macro_foreground_iou"])
print({k: res["meta"][k] for k in ("model", "trainable_params", "total_params", "input_size", "threshold", "device")})

### Python vs CLI
Same config and seed, so the two routes should agree (the tolerance only absorbs float non-determinism).

In [ ]:
cli_cats = {n: c["overall_foreground_iou"] for n, c in res["summary"]["categories"].items()}
py_cats = {n: c["overall_foreground_iou"] for n, c in py_summary["categories"].items()}
print("python:", py_cats, "\ncli   :", cli_cats)
assert all(abs(py_cats[n] - cli_cats[n]) < 1.0 for n in py_cats), "Python and CLI results diverged"

## 3 · Comparing with the SAM3 fine-tune (#5)
Both pipelines write `multiclass_foreground_iou_summary.json`, so the comparison is key-by-key on the **same `test/` split**:

```python
sam3 = json.load(open("<sam3_eval_dir>/multiclass_foreground_iou_summary.json"))["summary"]
samba = json.load(open("<sambaunet_eval_dir>/multiclass_foreground_iou_summary.json"))["summary"]
for name in sam3["categories"]:
    print(name, sam3["categories"][name]["overall_foreground_iou"], samba["categories"][name]["overall_foreground_iou"])
```

Compare `overall_foreground_iou`, `pixel_precision`, `pixel_recall` per category and `summary.overall`. Caveats: SAMBA-UNet has no text
prompts or instances (one sigmoid channel per category, 256 px input) and both trainers pick the best checkpoint on the test split,
so neither number is a clean held-out estimate. `meta.latency_ms_per_image` / `trainable_params` give the cost side.